# Colab GPU inference: Qwen Audio + AST

Run this notebook with a Colab **T4 or better GPU**. The runtime imports the integrated Python modules from `spatial-avla-pipeline`; `paper-diary` remains the research/training source and is cloned for provenance. Colab does not run the Docker services: both models use the same CUDA runtime. Qwen Audio is loaded lazily in 4-bit and analyzes uploaded-video audio windows asynchronously; AST continues to provide the measured template score.

Before running, put `mae_siren_templates_ast.pt` and `rtdetr-l.pt` in `MyDrive/paper-diary/models/`. The first file is the schema-v3 template generated for this pipeline; do not substitute `configs/siren_template.pt`.

In [ ]:
from pathlib import Path
import subprocess

WORK = Path('/content')
REPOSITORIES = {
    'spatial-avla-pipeline': 'https://github.com/todd-jang/spatial-avla-pipeline.git',
    'paper-diary': 'https://github.com/todd-jang/paper-diary.git',
}
for name, url in REPOSITORIES.items():
    target = WORK / name
    if not target.exists():
        subprocess.run(['git', 'clone', '--depth', '1', url, str(target)], check=True)
    print(f'{name}: {target}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

PIPELINE = WORK / 'spatial-avla-pipeline'
MODEL_ASSETS = Path('/content/drive/MyDrive/paper-diary/models')
required_assets = {
    MODEL_ASSETS / 'mae_siren_templates_ast.pt': PIPELINE / 'outputs/mae_siren_templates_ast.pt',
    MODEL_ASSETS / 'rtdetr-l.pt': PIPELINE / 'rtdetr-l.pt',
}
missing = [str(source) for source in required_assets if not source.is_file()]
if missing:
    raise FileNotFoundError('Upload required model assets to Drive first: ' + ', '.join(missing))
for source, target in required_assets.items():
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    print(f'Installed {source.name}: {target.stat().st_size:,} bytes')

In [ ]:
# Colab supplies CUDA-enabled torch/torchaudio. Install app/model dependencies without replacing them.
%pip install -q 'langgraph>=0.2.0' 'pydantic>=2.0.0' 'fastapi>=0.100.0' 'uvicorn>=0.20.0' 'gradio>=4.44.0' 'ultralytics>=8.3.0' 'transformers>=4.40.0' accelerate bitsandbytes soundfile scipy jsonschema yt-dlp

In [ ]:
import os
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is not available. In Colab choose Runtime > Change runtime type > T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)

os.chdir(PIPELINE)
sys.path.insert(0, str(PIPELINE))
os.environ['USE_QWEN_AUDIO'] = '1'
os.environ.setdefault('QWEN_AUDIO_MODEL', 'Qwen/Qwen-Audio-Chat')
os.environ.setdefault('GRADIO_SHARE', 'true')
os.environ.setdefault('GRADIO_SERVER_NAME', '0.0.0.0')
os.environ.setdefault('GRADIO_SERVER_PORT', '7860')

import torchaudio
from src.core.mae_vision import MaeVisionSirenScorer
scorer = MaeVisionSirenScorer(device='cuda')
if not scorer.available:
    raise FileNotFoundError('AST template was not installed into outputs/.')
print('AST template ready; torchaudio:', torchaudio.__version__)
print('Qwen Audio is enabled and will download/load on the first uploaded audio window.')

## Launch Gradio

The public share URL is created by Gradio for this session. Upload an MP4 with an audio track in the **Custom video upload** preset. AST template scores and RT-DETR detections are measured independently; Qwen Audio status/output appears in the tile after its background request completes. Qwen does not control or replace the calibrated AST result. Stop the cell to shut down the app.

In [ ]:
!python run_gradio.py